# Detecção de Anomalias com VLM

No notebook anterior (`04_01_deteccao_anomalias.ipynb`), usamos o **EfficientAD**, um modelo especializado e treinado apenas com imagens normais, para detectar anomalias em garrafas.

Neste notebook, vamos combinar as duas abordagens: usar o **EfficientAD** para localizar a anomalia (score, heatmap e máscara) e um **VLM** (Vision-Language Model), como em `03_03_vlm.ipynb`, para transformar essa detecção em um **laudo técnico** — como se estivéssemos pedindo a um inspetor de qualidade para redigir um parecer com base nas evidências do modelo especializado.

Em vez de perguntar ao VLM "existe uma anomalia nessa imagem?" (que ele teria que responder analisando a imagem sozinho, com risco de alucinação), vamos fornecer a ele a imagem original, o heatmap e o contorno gerados pelo `EfficientAD`, além do `score` calculado — para que o laudo seja fundamentado na detecção automática, e não em uma leitura visual "às cegas".

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install anomalib==2.5.1
    !pip install openai
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Vamos combinar as bibliotecas usadas em `04_01_deteccao_anomalias.ipynb` e em `03_03_vlm.ipynb`:
* `EfficientAd` / `Engine` / `PredictDataset`: para carregar o modelo especializado e gerar as previsões (score, heatmap, máscara).
* `openai`: para conversar com o VLM disponibilizado pela OpenRouter.
* `matplotlib`: para exibir e salvar as imagens de evidência.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from getpass import getpass
from pathlib import Path
import json
import base64
import mimetypes

from anomalib.data import PredictDataset
from anomalib.models import EfficientAd
from anomalib.engine import Engine

from openai import OpenAI

import torch

import numpy as np
import matplotlib.pyplot as plt
# Indica ao notebook to render figures in-page.
%matplotlib inline
from IPython.display import Markdown

## 1. Obtendo Chave da OpenRouter

Vamos utilizar a mesma **OpenRouter** apresentada em `03_03_vlm.ipynb` para acessar o VLM. Caso ainda não tenha uma chave, siga as instruções da seção 1.1 daquele notebook.

In [ ]:
API_KEY = getpass("Digite a API KEY do OpenRouter")
OPEN_ROUTER_DEFAULT_MODEL = 'google/gemma-4-26b-a4b-it:free'

client = OpenAI(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

### 1.1. Funções auxiliares

A função `image_to_data_url`, criada em `03_03_vlm.ipynb`, converte uma imagem em *data URL*. Já `perguntar_imagens` é uma adaptação da função `perguntar_imagem` daquele notebook, capaz de enviar **múltiplas imagens** (nesse caso, original + heatmap + contorno) em uma única mensagem para o VLM.

In [ ]:
def image_to_data_url(image_path: str) -> str:
    """
    Converte uma imagem para Data URL compatível com OpenAI/OpenRouter.

    Suporta:
        - jpg
        - jpeg
        - png
        - webp
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {image_path}"
        )

    mime_type, _ = mimetypes.guess_type(image_path)

    if mime_type not in {
        "image/jpeg",
        "image/png",
        "image/webp",
    }:
        raise ValueError(
            f"Formato não suportado: {mime_type}"
        )

    with open(image_path, "rb") as f:
        encoded = base64.b64encode(
            f.read()
        ).decode("utf-8")

    return f"data:{mime_type};base64,{encoded}"


def perguntar_imagens(
    image_paths: list,
    prompt: str,
    model: str = None
):
    content = [
        {
            "type": "text",
            "text": prompt
        }
    ]

    for image_path in image_paths:
        content.append({
            "type": "image_url",
            "image_url": {
                "url": image_to_data_url(image_path)
            }
        })

    response = client.chat.completions.create(
        model=model or OPEN_ROUTER_DEFAULT_MODEL,
        messages=[
            {
                "role": "user",
                "content": content
            }
        ]
    )

    return response.choices[0].message.content

## 2. Rodando o EfficientAD para gerar evidências

Vamos carregar o mesmo checkpoint usado em `04_01_deteccao_anomalias.ipynb` (`modelos/efficientad-bottle-epoch=19.ckpt`), treinado apenas com garrafas normais (`bottle`). Diferente daquele notebook, aqui vamos usar as saídas do modelo (`anomaly_map` e `pred_mask`) para gerar 3 imagens de evidência (original, heatmap e contorno), que serão fornecidas como contexto ao VLM na próxima seção.

In [ ]:
model = EfficientAd()

engine = Engine(
    accelerator="cpu",
    devices=1,
    enable_progress_bar=False,
    logger=False,
    default_root_dir="output/results"
)

checkpoint_path = "modelos/efficientad-bottle-epoch=19.ckpt"

### 2.1. Fazendo a predição na imagem com problema

Vamos usar a mesma imagem da seção 2 de `04_01_deteccao_anomalias.ipynb`: uma garrafa com quebra grande (`broken_large/000.png`).

In [ ]:
dataset = PredictDataset(path="imagens/04/bottle_test/broken_large/000.png")

predictions = engine.predict(
    model=model,
    dataset=dataset,
    ckpt_path=checkpoint_path
)

# predictions[0] é o ImageBatch (ainda com dimensão de lote); .items[0] devolve
# o ImageItem individual, sem a dimensão de lote, necessário para gerar_evidencias.
prediction_defeito = predictions[0].items[0]

print("Score:", prediction_defeito.pred_score)
print("Label:", prediction_defeito.pred_label)

### 2.2. Gerando as evidências visuais (original, heatmap e contorno)

A função `gerar_evidencias` extrai a imagem original, o mapa de calor da anomalia e o contorno da máscara prevista — os mesmos elementos exibidos em `apresentar_previsao` no notebook anterior —, salva cada um como um arquivo PNG separado e devolve seus caminhos, que serão usados como input para o VLM.

In [ ]:
def gerar_evidencias(item, saida_dir: str = "output/evidencias") -> dict:
    saida_dir = Path(saida_dir)
    saida_dir.mkdir(parents=True, exist_ok=True)

    # ------------------
    # Imagem
    # ------------------
    img = item.image

    if torch.is_tensor(img):
        img = img.detach().cpu().permute(1, 2, 0).numpy()

    img = np.clip(img, 0.0, 1.0)

    # ------------------
    # Anomaly Map
    # ------------------
    anomaly_map = item.anomaly_map

    if torch.is_tensor(anomaly_map):
        anomaly_map = anomaly_map.detach().cpu().numpy()

    anomaly_map = anomaly_map.squeeze()
    anomaly_map_norm = (anomaly_map - anomaly_map.min()) / (anomaly_map.max() - anomaly_map.min() + 1e-8)

    gray = img.mean(axis=2)
    obj_mask = gray < 0.95

    anomaly_map_masked = anomaly_map_norm.copy()
    anomaly_map_masked[~obj_mask] = np.nan

    # ------------------
    # Pred Mask
    # ------------------
    pred_mask = item.pred_mask

    if torch.is_tensor(pred_mask):
        pred_mask = pred_mask.detach().cpu().numpy()

    pred_mask = pred_mask.squeeze()

    nome = f"{Path(item.image_path).parent.name}_{Path(item.image_path).stem}"
    caminhos = {}

    # Original
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_original.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["original"] = str(caminho)

    # Heatmap
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.imshow(anomaly_map_masked, cmap="jet", alpha=0.6)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_heatmap.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["heatmap"] = str(caminho)

    # Contorno
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img)
    ax.contour(pred_mask, levels=[0.5], colors="red", linewidths=2)
    ax.axis("off")
    caminho = saida_dir / f"{nome}_contorno.png"
    fig.savefig(caminho, bbox_inches="tight", pad_inches=0)
    plt.close(fig)
    caminhos["contorno"] = str(caminho)

    # ------------------
    # Exibição conjunta
    # ------------------
    status = "ANOMALIA" if item.pred_label else "NORMAL"

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(plt.imread(caminhos["original"]))
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(plt.imread(caminhos["heatmap"]))
    axes[1].set_title("Heatmap")
    axes[1].axis("off")

    axes[2].imshow(plt.imread(caminhos["contorno"]))
    axes[2].set_title("Contorno")
    axes[2].axis("off")

    plt.suptitle(
        f"{Path(item.image_path).name} | {status} | score={float(item.pred_score):.3f}",
        fontsize=16,
        fontweight="bold"
    )
    plt.show()

    return caminhos

In [ ]:
caminhos_defeito = gerar_evidencias(prediction_defeito)
caminhos_defeito

## 3. Gerando um parecer técnico com o VLM

Agora vamos unir as duas abordagens: em vez de mandar apenas a imagem original para o VLM (como fizemos em `03_03_vlm.ipynb`), vamos enviar as 3 evidências geradas pelo `EfficientAD` (original, heatmap e contorno) e o `score` do modelo, para que o VLM redija o laudo com base no que o modelo especializado já detectou — em vez de "adivinhar" a anomalia sozinho.

### 3.1. Laudo em texto livre

In [ ]:
def montar_prompt_laudo(prediction) -> str:
    status_modelo = "ANOMALIA" if prediction.pred_label else "NORMAL"

    return f"""\
Você é um inspetor de controle de qualidade industrial especializado em garrafas de vidro.

Você recebeu três imagens da mesma garrafa, geradas por um modelo de detecção de anomalias (EfficientAD):

1. Imagem original: a garrafa inspecionada.
2. Mapa de calor (heatmap): quanto mais próximo do vermelho, maior a pontuação de anomalia atribuída pelo modelo àquela região.
3. Contorno: a região que o modelo aponta como defeito, destacada em vermelho.

O modelo especializado atribuiu a essa imagem um score de anomalia de {float(prediction.pred_score):.3f} (classificação: {status_modelo}).

Com base nessas evidências, produza um laudo técnico contendo:

1. **Objeto inspecionado**: o que é o objeto.
2. **Status**: NORMAL ou ANOMALIA (considere a classificação do modelo, mas use também seu próprio julgamento visual).
3. **Descrição da anomalia**: caso exista, descreva o que foi encontrado (tipo de defeito, formato, extensão), cruzando a imagem original com a região destacada no heatmap/ contorno.
4. **Localização**: em que região do objeto a anomalia se encontra (ex: base, lateral, gargalo).
5. **Gravidade**: BAIXA, MÉDIA ou ALTA.
6. **Recomendação**: o que deveria ser feito com essa peça (ex: aprovar, descartar, reinspecionar).

Responda em markdown, usando os tópicos acima.
"""

resposta = perguntar_imagens(
    [caminhos_defeito["original"], caminhos_defeito["heatmap"], caminhos_defeito["contorno"]],
    montar_prompt_laudo(prediction_defeito)
)

Markdown(resposta)

### 3.2. Laudo estruturado (JSON)

Assim como fizemos em `03_03_vlm.ipynb` para documentos, também podemos pedir ao VLM que devolva o laudo (agora considerando as evidências do `EfficientAD`) em um formato estruturado (JSON), o que facilita seu armazenamento em um banco de dados ou sua integração com outros sistemas. Incluímos ainda o campo `concorda_com_modelo_especializado`, que pode ser usado como sinalizador automático para revisão humana (HITL) sempre que o VLM discordar do modelo especializado.

In [ ]:
schema_laudo = {
    "type": "object",
    "properties": {

        "objeto_inspecionado": {
            "type": "string"
        },

        "status": {
            "type": "string",
            "description": "NORMAL ou ANOMALIA"
        },

        "tipo_anomalia": {
            "type": "string",
            "description": "Ex: quebra, trinca, contaminação. Vazio caso não haja anomalia"
        },

        "descricao": {
            "type": "string"
        },

        "localizacao": {
            "type": "string",
            "description": "Região do objeto onde a anomalia foi encontrada"
        },

        "gravidade": {
            "type": "string",
            "description": "BAIXA, MÉDIA ou ALTA"
        },

        "recomendacao": {
            "type": "string"
        },

        "concorda_com_modelo_especializado": {
            "type": "boolean",
            "description": "Se a análise do VLM concorda com a classificação do EfficientAD"
        }
    }
}

def montar_prompt_laudo_json(prediction) -> str:
    status_modelo = "ANOMALIA" if prediction.pred_label else "NORMAL"

    return f"""\
Você é um inspetor de controle de qualidade industrial especializado em garrafas de vidro.

Você recebeu três imagens da mesma garrafa, geradas por um modelo de detecção de anomalias (EfficientAD): a imagem original, o mapa de calor (heatmap) da anomalia e o contorno da região apontada como defeito.

O modelo especializado atribuiu a essa imagem um score de anomalia de {float(prediction.pred_score):.3f} (classificação: {status_modelo}).

Com base nessas evidências, extraia um laudo técnico sobre possíveis anomalias (defeitos) em JSON, usando a especificação abaixo delimitada entre ###.
###
{json.dumps(schema_laudo)}
###

Retorne apenas o JSON.
"""

resposta = perguntar_imagens(
    [caminhos_defeito["original"], caminhos_defeito["heatmap"], caminhos_defeito["contorno"]],
    montar_prompt_laudo_json(prediction_defeito)
)

Markdown(resposta)

## 4. Comparando com uma imagem normal

Vamos repetir o pipeline completo (predição + evidências + laudo) com uma garrafa sem defeito (`good/000.png`), para conferir se o VLM, mesmo recebendo o heatmap e o contorno do `EfficientAD`, identifica corretamente a ausência de anomalias.

In [ ]:
dataset = PredictDataset(path="imagens/04/bottle_test/good/000.png")

predictions = engine.predict(
    model=model,
    dataset=dataset,
    ckpt_path=checkpoint_path
)

prediction_normal = predictions[0].items[0]
caminhos_normal = gerar_evidencias(prediction_normal)

In [ ]:
resposta = perguntar_imagens(
    [caminhos_normal["original"], caminhos_normal["heatmap"], caminhos_normal["contorno"]],
    montar_prompt_laudo(prediction_normal)
)

Markdown(resposta)

## 5. VLM + Modelo Especializado: o melhor dos dois mundos

Neste notebook, unimos as duas abordagens em vez de escolher apenas uma:

- O **EfficientAD** localiza a anomalia com precisão pixel a pixel e calcula um `score` calibrado e consistente, sem "alucinar".
- O **VLM**, recebendo a imagem original, o heatmap, o contorno e o score como evidências, traduz essa detecção em um **laudo técnico legível**, com descrição, localização em linguagem natural, gravidade e recomendação — e, na versão em JSON, ainda indica se concorda com a classificação do modelo especializado.

Essa combinação tende a reduzir a alucinação do VLM (ele deixa de "adivinhar" a anomalia sozinho e passa a interpretar uma evidência concreta) e ainda aproveita a interpretabilidade do modelo especializado, que sozinho não consegue redigir um laudo. Em uma aplicação real, o campo `concorda_com_modelo_especializado` pode inclusive ser usado como um sinalizador automático para revisão humana (HITL): sempre que o VLM discordar do `EfficientAD`, o caso é encaminhado para um inspetor revisar.